In [1]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures 
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import FuncProcessor, FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector, AnalysisFlowConfig, AnalysisFlow, IoSMetricsCalculatorConfig
from vistiq.core import labels_to_masks
from vistiq.analysis.overlap import (
    OverlapCalculator,
    LabelOverlapCalculatorConfig,   # or BoxOverlapCalculatorConfig / MaskOverlapCalculatorConfig
    IoSMetricsCalculatorConfig,
    metrics_calculator_configs,
    region_map_from_dataframe,      # if using region maps from DataFrames
)
from vistiq.analysis import MatrixAggregator, MatrixAggregatorConfig, MatrixCombiner, MatrixCombinerConfig
from vistiq.constant.matrix import UPPER
from vistiq.constant import LOWER_ND
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig
from vistiq.graph import NXGraphBuilder, NXGraphBuilderConfig, NXGraphQuery, NXGraphQueryConfig

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results

import stackview
import os
import copy
import numpy as np
import math
import logging
import pandas as pd
import itertools

from typing import Any, List, Tuple
from pathlib import Path

2026-06-17 22:30:33,913 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


# Configure logger and check availability of accelerators

In [2]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

2026-06-17 22:30:40,009 - INFO - Found mps device: Apple Metal (MPS)
2026-06-17 22:30:40,010 - INFO - Available Torch accelerators: mps


# Load image

In [3]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [4]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    substack="Z:20-50"
)
img, metadata = ImageLoader(ilc).run(path)

2026-06-17 22:30:40,700 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-17 22:30:40,766 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-17 22:30:40,770 - INFO - Loading image from: Animal 1.lif
2026-06-17 22:30:41,912 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-06-17 22:30:41,913 - INFO - Applying substack={'Z': slice(19, 50, None)}
2026-06-17 22:30:42,019 - INFO - Loaded image: Animal 1.lif scene=0 -> shape=(3, 31, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-06-17 22:30:42,020 - INFO - Loaded image with shape: (3, 31, 512, 512), dtype: uint8
2026-06-17 22:30:42,021 - INFO - Finished in state Completed()


In [5]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [6]:
tissue_ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
#c_img, c_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
#metadata, c_metadata

In [7]:
# stackview.slice(c_img)

# Configuration for 3D Tissue Segmentation

In [8]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0)
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)

# Configuration for Region Analysis

In [9]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)


# Configuration for Cell Segmentation

In [10]:
# specify preprocessing config for cells
cell_ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [11]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
        )
    ]
)

cell_sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [12]:
@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [13]:
acfg = AnalysisFlowConfig(
    region_analyzer = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe",
        index_on="object_id",
        map_axes=True,
    ),
    #coincidence_detector = CoincidenceDetectorConfig(
    #    method=IoSMetricsCalculatorConfig(),
    #    iterator_config=ArrayIteratorConfig(slice_def=()),
    #    mode="outline",
    #),
    overlap_calculator = LabelOverlapCalculatorConfig(
        metrics_calculators = [IoSMetricsCalculatorConfig()],
        output_type="dataframe",
        annotate=True,
        triangle=7,
    ),
    overlap_filter = ValueFilterConfig(
        ref_value=0.5,
        axis=0,
        operator=">",
        triangle=LOWER_ND,
        output="masked_values",
    ),
    overlap_aggregator = MatrixAggregatorConfig(
        operation="count",
        axis=1,
    ),
)

In [14]:
@flow
def full_pipeline(img_path, scene_index=0, outdir=".", embedding_path="embeddings"):
    # load image
    img, metadata = ImageLoader(ilc).run(img_path)
    
    # TISSUE - brain lobes
    # preprocess
    tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
    tissue_metadata = copy.deepcopy(tissue_metadata)
    tissue_metadata["channel_names"] = ["Lobe"]
    # segment tissue
    tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)

    # BRAIN - all tissue combined
    #tissue_masks = labels_to_masks(tissue_labels)
    brain_label = (tissue_labels>0).astype("uint16")
    brain_metadata = copy.deepcopy(tissue_metadata)
    brain_metadata["channel_names"] = ["Brain"]
    
    # CELLS
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(cell_ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    cell_labels = SegmentationFlow(cell_sfcfg).mapped_run(channels, metadata=channel_metadata)
    
    # Analyze CELLS and TISSUE
    # analyze regions in each channel separately
    combined_labels = [brain_label, tissue_labels, *cell_labels]
    combined_metadata = [brain_metadata, tissue_metadata, *channel_metadata]
    measurements = AnalysisFlow(acfg).run(combined_labels, metadata=combined_metadata)
    # measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*channel_metadata, c_metadata, b_metadata])

    # save labels
    fname_stem = Path(img_path).stem
    imc = ImageWriterConfig(overwrite=True)
    outpaths = [os.path.join(outdir, f'{fname_stem}.scene-{meta.get("scene_index","")}.tif') for meta in combined_metadata]
    # print (outpaths)
    ImageWriter(imc).run.map(combined_labels, outpaths, metadata=combined_metadata)
    
    # make sure to resolve the futures to results
    return (
        resolve_futures(combined_labels),
        resolve_futures(combined_metadata),
        resolve_futures(measurements),
    )


In [15]:
combined_labels, combined_metadata, measurements = full_pipeline(path, scene_index=scene_index, outdir=".", embedding_path=embedding_path)

2026-06-17 22:30:42,865 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-17 22:30:43,108 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-17 22:30:43,467 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a33585-2fc1-726a-8000-566e83ad2ff9/set_state "HTTP/1.1 201 Created"
2026-06-17 22:30:43,552 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a33585-2fc1-726a-8000-566e83ad2ff9 "HTTP/1.1 200 OK"
2026-06-17 22:30:43,613 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

Using apple MPS device.


2026-06-17 22:30:52,389 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-17 22:30:55,451 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-17 22:30:55,452 - INFO - StackProcessor.run: received workers=2 (type: <class 'int'>)

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-17 22:31:02,390 - INFO - Running OverlapCalculator with config: classname='Configurable' package='vistiq.core' version=None command_group=None builder=MaskStackBuilderConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, preferred_input_type='torch.Tensor', preferred_device=None) area_calculator=MaskAreaCalculatorConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, preferred_input_type='torch.Tensor', preferred_device=None) intersection_calculator=MaskIntersectionCalculatorConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, preferred_input_type='torch.Tensor', preferred_device=None, memory_limit_mb=5120, prune_bboxes=False, dense_pair_fraction=1.01) metrics_calculators=[IoUMetricsCalculatorConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, name='iou')] return_components=False triangle=2 output_type='np.ndarray' annotate=False
2026

...

2026-06-17 22:31:08,988 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-17 22:31:17,759 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-17 22:31:17,760 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>
DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-17 22:31:34,640 - INFO - Finished in state Completed()
2026-06-17 22:31:34,655 - INFO - Creating RegionAnalyzer for region filter with properties: ['label', 'object_id', 'slice_id', 'stack_id', 'centroid', 'cross_sectional_area-xy']
2026-06-17 22:31:34,735 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None index_on='label' properties=['label', 'object_id', 'slice_id', 'stack_id', 'centroid', 'cross_sectional_area-xy'] map_axes=False expand_coordinates=False
2026-06-17 22:31:34,735 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-17 22:31:34,749 - INFO - RegionAnalyzer: Applying scale: (-0.9999284782608696, 0.300

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-17 22:31:35,476 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a33586-a373-7d8f-8000-bb925e163974/set_state "HTTP/1.1 201 Created"
2026-06-17 22:31:35,570 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-06-17 22:31:36,108 - INFO - Finished in state Completed('All states completed.')
2026-06-17 22:31:36,335 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/task_runs/ "HTTP/1.1 201 Created"
2026-06-17 22:31:36,455 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/filter "HTTP/1.1 200 OK"
2026-06-17 22:31:36,623 - INFO 

In [16]:
#combined_metadata

In [17]:
stackview.slice(np.concatenate(combined_labels, axis=-1))

In [18]:
for k,v in measurements.items():
    print (k, type(v))
df=measurements["region_analyzer_all"].sort_values(["channel","label"])
df

region_analyzer: Brain <class 'pandas.core.frame.DataFrame'>
region_analyzer: Lobe <class 'pandas.core.frame.DataFrame'>
region_analyzer: Scrib <class 'pandas.core.frame.DataFrame'>
region_analyzer: EdU <class 'pandas.core.frame.DataFrame'>
region_analyzer: Dpn <class 'pandas.core.frame.DataFrame'>
overlap: Brain vs Lobe <class 'pandas.core.frame.DataFrame'>
overlap: Brain vs Scrib <class 'pandas.core.frame.DataFrame'>
overlap: Brain vs EdU <class 'pandas.core.frame.DataFrame'>
overlap: Brain vs Dpn <class 'pandas.core.frame.DataFrame'>
overlap: Lobe vs Brain <class 'pandas.core.frame.DataFrame'>
overlap: Lobe vs Scrib <class 'pandas.core.frame.DataFrame'>
overlap: Lobe vs EdU <class 'pandas.core.frame.DataFrame'>
overlap: Lobe vs Dpn <class 'pandas.core.frame.DataFrame'>
overlap: Scrib vs Brain <class 'pandas.core.frame.DataFrame'>
overlap: Scrib vs Lobe <class 'pandas.core.frame.DataFrame'>
overlap: Scrib vs EdU <class 'pandas.core.frame.DataFrame'>
overlap: Scrib vs Dpn <class 'pand

,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,centroid-z,centroid-y,centroid-x,label,...,object_name,count Dpn,count EdU,count Lobe,count Scrib,lineage Brain,lineage Dpn,lineage EdU,lineage Lobe,lineage Scrib
object_id,,,,,,,,,,,,,,,,,,,,,
5773d5fd65b94620a859254931293715,0,0,52,31,508,468,-16.049514,82.509963,75.001211,1,...,Brain 1,90.0,183.0,2.0,147.0,NaN,NaN,NaN,NaN,NaN
9c1e77a1c81d435694b1f75c7cbc0f8e,0,90,177,7,113,192,-2.715642,30.322650,55.131900,1,...,Dpn 1,NaN,NaN,NaN,NaN,1.0,NaN,15.0,1.0,NaN
2fb607309f6d4e66839342c041090f45,0,108,200,6,137,223,-2.520354,36.195326,63.371667,2,...,Dpn 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90c6d222b6b64cdf96f5cff0f34cac63,0,117,221,9,138,246,-3.477915,38.225660,70.301012,3,...,Dpn 3,NaN,1.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN
fd041735bcda4bd0b84e256408228639,0,149,234,9,170,258,-3.942043,47.915481,73.290544,4,...,Dpn 4,NaN,1.0,NaN,NaN,1.0,NaN,NaN,1.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
003df068dbac4f1089a5481483b5b79c,30,344,411,31,391,460,-29.997854,109.242558,131.795382,243,...,Scrib 243,NaN,1.0,NaN,NaN,1.0,NaN,NaN,2.0,NaN
b61734a2a3bc4109b27adadf1ffaf238,30,363,249,31,415,326,-29.997854,116.698411,85.848485,244,...,Scrib 244,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2.0,NaN
3f2da7f7c4f84cac80fdbbf326eb55a3,30,384,158,31,415,176,-29.997854,119.526690,49.991774,245,...,Scrib 245,1.0,NaN,NaN,NaN,1.0,NaN,NaN,2.0,NaN


# Query graph for object ancestor lineage

1. Subcellular cellular Dpn -> tissue lobe -> organ brain
2. Count descendants in each channel

In [20]:
dag = measurements["containment_graph"]

gqcfg = NXGraphQueryConfig(
    attributes=["descendant_counts", "ancestor_lineage"],
    filter_attribute="channel",
    filter_value="Lobe",
    include_attributes=["label", "channel"],
    lineage_value_attribute="label",
    output_type="dataframe",
)
gq = NXGraphQuery(gqcfg)
result = gq.run(dag, node=None)
df_counts = gq.format(result["descendant_counts"])
df_lineage = gq.format(result["ancestor_lineage"])
df = pd.concat([df_counts, df_lineage], axis=1)
df = df.loc[:, ~df.columns.duplicated()].sort_values(["label"])
df

2026-06-17 23:05:58,668 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-17 23:05:58,715 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-17 23:05:58,727 - INFO - Summarizing graph with config: classname='Configurable' package='vistiq.core' version=None command_group=None label_attribute='object_name' group_attribute='channel' filter_attribute='channel' filter_value='Lobe' include_attributes=['label', 'channel'] lineage_value_attribute='label' attributes=['descendant_counts', 'ancestor_lineage'] output_type='dataframe' output_index='object_id'
2026-06-17 23:05:58,727 - INFO - Node: None
2026-06-17 23:05:58,732 - INFO - Finished in state Completed()
2026-06-17 23:05:58,847 - INFO - HTTP Request

,channel,label,count Dpn,count EdU,count Scrib,lineage Brain
object_id,,,,,,
e1ad228f9f7f48f7885b100e4ab20eff,Lobe,1,45,93,73,1
11d89d26ccd04f03a58753bb4279c260,Lobe,2,45,90,74,1


In [77]:
from pyvis.network import Network

net = Network(notebook=True, cdn_resources='in_line', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(dag)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
for node in net.nodes:
    node["title"] = node["object_name"] #+ "\n" +"  Neighbors:\n" + "\n".join(neighbor_map[node["id"]])
    #node["value"] = len(neighbor_map[node["object_name"]])

# Render
net.show("nx_graph.html")

nx_graph.html


In [ ]:
net.show_buttons(filter_=['physics'])

# Hierarchical label decomposition

# View in Napari

In [ ]:
import napari
viewer = napari.Viewer()

In [ ]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
nimg = img#np.expand_dims(img, axis=0)

# add brain label
viewer.add_labels(brain, name="Brain", scale=scale)

# add lobe labels
for ch, l, m in zip(metadata["channel_names"],cell_labels, cell_measurements):
    new_m = m.copy().reset_index()
    background = pd.DataFrame({c: [0] if c=="label" else [np.nan] for c in new_m.columns.to_list()})
    new_m = pd.concat([background, new_m], ignore_index=True)
    # print (new_m)
    viewer.add_labels(l, name=f"{ch}-Labels", features=new_m, scale=scale)

# add cell labels for each channel
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{name}", scale=scale, colormap=color, blending="additive")
    


In [ ]:
dl1 = viewer.layers["Dpn-Labels-Lobe 1"]
dl2 = viewer.layers["Dpn-Labels-Lobe 2"]
print (np.intersect1d(np.unique(dl1.data), np.unique(dl2.data)))
